# Demodulacion GMSK sobre Red Pitaya -- usando la libreria

Genera una señal de prueba tipo AIS (con NRZI, sin bit stuffing), la
transmite por la Red Pitaya en loopback, la captura, y la demodula
usando **solo** las funciones/clases de `gmsk_demod` -- el resultado es
un stream de bits sincronizado, sin ninguna logica de framing/CRC.

Usa las tres librerias:
- `gmsk_tx_signal`: arma y modula la señal de prueba (lado TX).
- `redpitaya_io`: habla con el hardware (protocolo de la Red Pitaya).
- `gmsk_demod`: demodulacion pura (lo que de verdad importa aca).

In [1]:
# ============================================================
# CELDA 0 — Imports y parametros de radio
# ============================================================
import sys
sys.path.insert(0, '.')   # asume que los .py de la libreria estan en esta carpeta

import numpy as np

import gmsk_tx_signal as tx
import redpitaya_io as rpio
import gmsk_demod as demod

# ---------------- parametros de radio ----------------
FS      = 125e6                 # tasa del DAC/ADC
fs_out  = 125e6 / 651            # tasa de salida del receptor (~192.012 ksps)
f_lo    = 10.7e6                 # LO del receptor
OFFSET  = 0e3                    # offset de portadora sobre el LO
BAUD    = 9600.0                 # AIS: 9600 bit/s
BT      = 0.4                    # AIS: BT nominal 0.4 (ITU-R M.1371)
L_PULSO = 4                      # duracion del pulso en simbolos

AMPLITUD = 4000
TX_BURST_SAMPLES = 32

NRZI_HABILITADO = True

SPS_TX = round(FS / BAUD)        # muestras/simbolo en el DAC
SPS_RX = fs_out / BAUD           # muestras/simbolo en el receptor

HOST = '192.168.1.100'
PORT = 1003

N_RX = 12000   # muestras complejas por captura

print(f"NRZI_HABILITADO = {NRZI_HABILITADO}")
print(f"SPS_TX = {SPS_TX}   SPS_RX = {SPS_RX:.4f}")


NRZI_HABILITADO = True
SPS_TX = 13021   SPS_RX = 20.0013


In [2]:
# ============================================================
# CELDA 1 — Generar la señal de prueba (libreria gmsk_tx_signal)
# ============================================================
rng = np.random.RandomState(2024)
bits_trama, payload_tx = tx.generar_trama_bits(rng, NRZI_HABILITADO)
words_tx, fc_adj, N_buf, bb_ranura = tx.construir_buffer_dac(
    bits_trama, FS, f_lo, OFFSET, SPS_TX, L_PULSO, BT,
    AMPLITUD, TX_BURST_SAMPLES, NRZI_HABILITADO)
CFO_REAL = fc_adj - f_lo

print(f"Buffer TX: {N_buf} muestras, {N_buf/SPS_TX:.1f} simbolos")
print(f"fc = {fc_adj/1e6:.6f} MHz (offset {CFO_REAL/1e3:.3f} kHz sobre LO)")


Buffer TX: 2916704 muestras, 224.0 simbolos
fc = 10.700006 MHz (offset 0.006 kHz sobre LO)


In [3]:
# ============================================================
# CELDA 2 — Conexion con la Red Pitaya (libreria redpitaya_io)
# ============================================================
cfg_reset, cfg_run = rpio.calcular_pinc_y_cfg(f_lo, FS)
sock = rpio.abrir_y_arrancar(HOST, PORT, words_tx, cfg_reset, cfg_run)
print(f"Conectado a {HOST}:{PORT}, TX corriendo en loop continuo.")
print(f"Captura de {N_RX} muestras = {N_RX/fs_out*1e3:.1f} ms "
      f"({N_RX/fs_out*BAUD:.0f} simbolos)")


Conectado a 192.168.1.100:1003, TX corriendo en loop continuo.
Captura de 12000 muestras = 62.5 ms (600 simbolos)


In [4]:
# ============================================================
# CELDA 3 — Captura y demodulacion pura (libreria gmsk_demod)
# ============================================================
iq = rpio.read_adc(sock, N_RX)
iq = iq / (np.max(np.abs(iq)) + 1e-12)

rx = demod.ReceptorGmskCiego(iq, fs_out, SPS_RX)
rx.run()

print(f"CFO estimado: {rx.offset_est/1e3:.3f} kHz "
      f"(objetivo {CFO_REAL/1e3:.3f} kHz, error {(rx.offset_est-CFO_REAL):.1f} Hz)")

bits_sync = rx.bits_sincronizados(nrzi_habilitado=NRZI_HABILITADO)
print(f"Stream de bits sincronizado: {len(bits_sync)} bits")
print(bits_sync[:64])


CFO estimado: -0.123 kHz (objetivo 0.006 kHz, error -129.1 Hz)
Stream de bits sincronizado: 599 bits
[1 1 1 0 1 1 0 1 0 1 1 1 1 1 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 1 0 1 0 1
 1 1 0 0 1 1 0 1 0 1 1 0 1 1 0 0 0 0 0 1 1 1 0 1 1 1 1]


## Verificación

Compara el stream sincronizado contra la trama conocida por
correlacion, para confirmar que la demodulacion anda.

In [5]:
# ============================================================
# CELDA 4 — Verificacion de la captura (fuera de la libreria de demod)
# ============================================================
ref = tx.nrzi_encode(np.tile(np.array(bits_trama, dtype=np.uint8), 5)) \
      if NRZI_HABILITADO else np.tile(np.array(bits_trama, dtype=np.uint8), 5)

# para comparar contra bits_sync (ya decodificado), tambien se decodifica
# la referencia una vez armada la version cruda -- o, mas simple, se
# compara la version CRUDA de bits_sync contra la referencia codificada.
_, symbols0, _ = rx.bits_con_tau(demod.recuperar_fase_simbolo(rx.soft(), SPS_RX))
tau0 = demod.recuperar_fase_simbolo(rx.soft(), SPS_RX)
bits_crudos, _, _ = rx.bits_con_tau(tau0)

bc = bits_crudos.astype(np.int8)
n = len(bc)
mejor_offset, mejor_score = None, -1
for off in range(len(ref) - n):
    seg = ref[off:off+n].astype(np.int8)
    score = max(int(np.sum(seg == bc)), int(np.sum(seg != bc)))
    if score > mejor_score:
        mejor_score, mejor_offset = score, off

print(f"Mejor alineacion: offset={mejor_offset}, "
      f"coincidencia={100*mejor_score/n:.1f}% ({mejor_score}/{n} bits crudos)")


Mejor alineacion: offset=66, coincidencia=97.7% (585/599 bits crudos)


In [6]:
# ============================================================
# CELDA 5 — Cierre
# ============================================================
rpio.tx_stop(sock)
sock.close()
print("TX detenido, conexion cerrada.")


TX detenido, conexion cerrada.
